# §35 (v3) — Zincirleme: kapasite × yazım kuralı (küçük ölçek)

**Otorite ön-kayıt `RESULTS.md` §35'tedir.** Bu markdown yalnızca kolaylık kopyasıdır.

**Merkezi soru:** state neden **bir** sınırı geçiyor da **ikincisini** geçemiyor?

**Hipotez — projenin kendi bulgusundan:** bellek **girişim-sınırlı** (README §0;
§27a'da η taramasıyla bağımsız doğrulandı — platoyu uzatmak girişimi biriktirip
*kötüleştirdi*). Dolgu chunk'ları her `CC_DIST_EVERY` tokende bir distraktör kv
yazıyor; iki chunk sonra hedef gömülüyor.

---

## v3 — neden tasarım değişti (GÜÇ)

v2 tasarımı 2×2 kol × **2 seed**, birincil metrik K=2'de sonda doğruluğu, mutlak
eşik %30 idi. Bu yorumlanamaz:

- §26b **6 seed / 200 deneme** ile aynı aracı koştu ve K=0'da exp kolu
  **%33.2, seed aralığı %8.5–69.5** çıktı. Kendi kaydımız bunu *yetersiz güç →
  sonuçsuz* diye kapattı ve dersi yazdı: "12–20 seed **ya da** düşük-varyanslı
  bir endpoint gerekir."
- v2 seed sayısını 2'ye indirip eşiği mutlak bıraktı: güç §26b'den **daha düşük**,
  hüküm ise daha kesin. Beklenen sonuç "ETKİ YOK" — ama bu, girişim hipotezinin
  yanlışlığından değil örnek sayısından gelirdi. Projenin **güç kontrolü** kuralı
  (iki kol da şanstaysa "null" değil "sonuçsuz") tam da bunu yasaklıyor.

**v3 düzeltmesi (bütçe-nötr: yine 8 koşu):**

| | v2 | v3 |
|---|---|---|
| kol | 4 (2×2) | **2** (taban vs en güçlü kontrast) |
| seed | 2 | **4** |
| birincil metrik | K=2 sonda doğruluğu (%) | **cross-chunk doğrulama kaybı** (nat, eşleşmiş) |
| eşik | mutlak %30 | **eşleşmiş Δ ≤ −0.15 nat + işaret tutarlılığı** |

0.15 nat eşiği §28a'nın kendi çürütme eşiğinden alındı (keyfi değil).

**Kolların birleştirilmesinin bedeli (açıkça):** taban `nu=2/additive` ile
`nu=4/delta` karşılaştırılıyor; sinyal çıkarsa **hangi kaldıracın** (kapasite mi
delta mı) sorumlu olduğu bu koşudan bilinemez, ayrıştırma bir sonraki adımdır.
Bu bilinçli bir takas: ayrıştırmadan önce **bir etki tespit edebilmek** gerekiyor.

**Bu bir TARAMA (screening) deneyidir, doğrulayıcı değil.** n=4 eşleşmiş seed'de
işaret testi en iyi ihtimalle p=0.0625 verir; "kanıtlandı" denmeyecek.

In [ ]:
# --- 1. KURULUM ---
import os, subprocess, sys, re, json, itertools
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE,'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
CKDIR = os.path.join(BASE,'chain35'); os.makedirs(CKDIR, exist_ok=True)
try:
    from google.colab import drive; drive.mount('/content/drive')
    CKDIR = '/content/drive/MyDrive/hfp_chain35'; os.makedirs(CKDIR, exist_ok=True)
except Exception as e:
    print(f'Drive yok ({type(e).__name__}) -> yerel: {CKDIR}')
import torch
DEV_OK = torch.cuda.is_available()
print('repo:', REPO, '| ckpt:', CKDIR)
print('GPU:', torch.cuda.get_device_name(0) if DEV_OK else 'YOK (CPU)')
if not DEV_OK:
    print('  UYARI: CPU\'da bu tarama cok uzun surer (v1 olcumu ~25 saatti). Kaggle: Accelerator > T4 sec.')
print('  NOT: Bu notebooku AYNI ANDA iki kez calistirma — loglar cift basar,')
print('       iki surec ayni checkpointlere yazar ve her sey 2x yavaslar.')
for nu in (2,4): print(f'  dpfp_nu={nu} -> key_dim={2*64*nu}, M=({2*64*nu},64)={2*64*nu*64:,} float/katman')

In [ ]:
# --- 1b. ON-UCUS (preflight): ~1 dk. GERCEK KOSUYA BASLAMADAN ONCE.
# Amac: CI bu scripti KOSMUYOR. Burada 2 adimlik sahte bir kosuyla
# (a) script ayakta mi, (b) [§35] birincil metrik CSV'si yaziliyor mu
# dogrulanir. AYRI bir ckpt klasoru kullanir -> gercek kosunun
# checkpointlerini/atlama mantigini KIRLETMEZ.
import tempfile, csv as _csv
PRE = os.path.join(tempfile.gettempdir(), 'hfp_preflight35')
os.makedirs(PRE, exist_ok=True)
_penv = {**os.environ, 'PYTHONPATH': REPO, 'HFP_CKPT_DIR': PRE,
         'CC_STEPS':'2','CC_CTX':'64','CC_P':'2','CC_BS':'2','CC_CARRY_MAX':'2',
         'CC_DIST_EVERY':'32','CC_GAPS':'256','CC_TRIALS':'3',
         'CC_VAL_N':'16','CC_VAL_K':'2','CC_DPFP_NU':'2','CC_WRITE':'additive'}
_r = subprocess.run([sys.executable,'review_scripts/carry_curriculum.py','exp','0','300'],
                    cwd=REPO, env=_penv)
_f = os.path.join(PRE, 'carryv1_exp_s0_valloss.csv')
assert _r.returncode == 0, f'ON-UCUS BASARISIZ: script cikis kodu {_r.returncode}. GERCEK KOSUYU BASLATMA.'
assert os.path.exists(_f), f'ON-UCUS BASARISIZ: birincil metrik CSV yazilmadi ({_f}). GERCEK KOSUYU BASLATMA.'
_row = next(_csv.DictReader(open(_f)))
_need = {'mode','seed','dpfp_nu','write','val_k','val_n','val_loss','val_sem'}
assert _need <= set(_row), f'ON-UCUS BASARISIZ: CSV alanlari eksik: {_need - set(_row)}'
assert int(_row['val_n']) == 16 and int(_row['val_k']) == 2, f'ON-UCUS BASARISIZ: {_row}'
float(_row['val_loss']); float(_row['val_sem'])
print(f'\nON-UCUS TAMAM -> {_row}')
print('(val_loss ~3.4 = sans; 2 adim egitilmis model icin BEKLENEN. Onemli olan')
print(' sayinin degeri degil, dosyanin yazilmasi ve alanlarin dogru olmasi.)')


In [ ]:
# --- 2. KOLLAR: 2 kol x 4 seed (v3: guc icin kol sayisi seede takas edildi) ---
# [v2 KAPSAM KISINTISI, korunuyor] CTX 256->128 | STEPS 1200->600 | BS 8->4
#   | CARRY_MAX 16->8. Taban kol AYNI kosuda oldugu icin ic karsilastirma korunur.
# [v3] ARMS 4->2, SEEDS 2->4. Toplam kosu sayisi degismedi (8).
SEEDS = [0,1,2,3]
BASE_ARM  = (2,'additive','taban (nu2/additive)')
TREAT_ARM = (4,'delta',   'nu4 + delta')
ARMS = [BASE_ARM, TREAT_ARM]
BASE_ENV = {**os.environ, 'PYTHONPATH': REPO, 'HFP_CKPT_DIR': CKDIR,
            'CC_CARRY_MAX':'8','CC_STEPS':'600','CC_CTX':'128','CC_P':'4',
            'CC_DIST_EVERY':'32','CC_BS':'4','CC_GAPS':'256','CC_TRIALS':'30',
            # [v3 BIRINCIL METRIK] egitim-sonu cross-chunk dogrulama kaybi,
            # K=2'de, 64 FARKLI ornek uzerinde ortalanir ve CSV'ye yazilir.
            'CC_VAL_N':'64','CC_VAL_K':'2'}
MODE = 'exp'                      # tek degisken kapasite/yazim olsun (retention sabit)
def cfgtag(nu, wr):
    return '' if (nu==2 and wr=='additive') else f'_nu{nu}{wr[0]}'
for nu, wr, label in ARMS:
    for s in SEEDS:
        cfg = cfgtag(nu, wr)
        if os.path.exists(f'{CKDIR}/carryv1{cfg}_{MODE}_s{s}_valloss.csv'):
            print(f'[atla] {label} s{s}'); continue
        print(f'\n=== EGITIM {label} (nu={nu}, {wr}) s{s} ===', flush=True)
        env = {**BASE_ENV, 'CC_DPFP_NU': str(nu), 'CC_WRITE': wr}
        subprocess.run([sys.executable,'review_scripts/carry_curriculum.py',MODE,str(s),'3000'],
                       cwd=REPO, env=env)
print('\nEGITIM TAMAM')

In [ ]:
# --- 3. IKINCIL: eslesmis sonda, K in {0,1,2,4} ---
# Sonda ARTIK BIRINCIL DEGIL (§26b: seed-basi %8.5-69.5 yayilim). Betimleyici.
import csv
KS = [0,1,2,4]
MP_TRIALS = 50
for nu, wr, label in ARMS:
    for s in SEEDS:
        cfg = cfgtag(nu, wr)
        if not os.path.exists(f'{CKDIR}/carryv1{cfg}_{MODE}_s{s}.pt'):
            print(f'[eksik ckpt] {label} s{s}'); continue
        if os.path.exists(f'{CKDIR}/matchedv1{cfg}_{MODE}_s{s}.csv'):
            continue
        env = {**BASE_ENV, 'MP_DPFP_NU': str(nu), 'MP_WRITE': wr,
               'MP_KS': ','.join(map(str,KS)), 'MP_TRIALS': str(MP_TRIALS),
               'MP_CTX':'128', 'MP_P':'4', 'MP_DIST_EVERY':'32'}
        print(f'\n=== EVAL {label} s{s} ===', flush=True)
        subprocess.run([sys.executable,'review_scripts/matched_probe.py',MODE,str(s)],
                       cwd=REPO, env=env)
print('\nEVAL TAMAM')

In [ ]:
# --- 4. BIRINCIL: eslesmis cross-chunk dogrulama kaybi + ON-KAYITLI HUKUM (§35 v3) ---
import csv, math, statistics as st

def val_loss(nu, wr, s):
    f = f'{CKDIR}/carryv1{cfgtag(nu,wr)}_{MODE}_s{s}_valloss.csv'
    if not os.path.exists(f): return None
    r = next(csv.DictReader(open(f)))
    return float(r['val_loss'])

print('=== BIRINCIL: cross-chunk dogrulama kaybi (nat, K=2, n=64 ornek/seed) ===')
print(f"{'seed':>6} {'taban':>10} {'nu4+delta':>12} {'delta(fark)':>13}")
print('-'*45)
diffs, pairs = [], []
for s in SEEDS:
    b = val_loss(*BASE_ARM[:2], s); t = val_loss(*TREAT_ARM[:2], s)
    if b is None or t is None:
        print(f'{s:>6} {"—":>10} {"—":>12} {"eksik":>13}'); continue
    d = t - b; diffs.append(d); pairs.append((s,b,t))
    print(f'{s:>6} {b:>10.4f} {t:>12.4f} {d:>+13.4f}')
print('(sans ~3.40 nat; DUSUK kayip = iyi)')

print('\n=== ON-KAYITLI HUKUM (§35 v3 — TARAMA, dogrulayici degil) ===')
if len(diffs) < len(SEEDS):
    print(f'  EKSIK VERI: {len(diffs)}/{len(SEEDS)} seed. Hukum verilmez.')
else:
    md = st.mean(diffs)
    sd = st.stdev(diffs) if len(diffs) > 1 else float('nan')
    sem = sd/math.sqrt(len(diffs)) if len(diffs) > 1 else float('nan')
    n_better = sum(1 for d in diffs if d < 0)
    t_stat = md/sem if sem and sem == sem and sem > 0 else float('nan')
    print(f'  eslesmis ortalama D = {md:+.4f} nat  (SD {sd:.4f}, SEM {sem:.4f}, n={len(diffs)})')
    print(f'  isaret: {n_better}/{len(diffs)} seed tedavi lehine | eslesmis t = {t_stat:+.2f}')
    print(f'  NOT: n={len(diffs)} eslesmis seedde isaret testi en iyi ihtimalle p=0.0625 verir.')
    if md <= -0.15 and n_better == len(diffs):
        print('\n  => SINYAL VAR: girisim kaldiraci cross-chunk ogrenmeyi olculebilir')
        print('     sekilde iyilestiriyor. "Kanitlandi" DEGIL — tarama gecti.')
        print('     Sonraki (tek): ayristirma + guc — nu4/additive vs nu2/delta,')
        print('     8-12 seed, ayni birincil metrik. Sonda dogrulugu ikincil kalir.')
    elif md >= 0.15 and n_better == 0:
        print('\n  => TERS YON: kapasite+delta cross-chunk ogrenmeyi KOTULESTIRIYOR.')
        print('     Girisim hipotezi bu yonde curudu; hat kapanir.')
    else:
        print('\n  => SONUCSUZ (null DEGIL): etki 0.15 nat esiginin altinda ya da')
        print('     isaretler bolunmus. Bu guc seviyesinde girisim hipotezi ne')
        print('     dogrulanir ne curutulur. Proje kurali geregi "etki yok" YAZILMAZ.')
        print('     Karar Kayrahan\'in: (a) ayni kolu 8-12 seede cikar, ya da')
        print('     (b) hatti park et ve yayin/okuma-yolu teshisine gec.')

In [ ]:
# --- 5. IKINCIL (betimleyici): eslesmis sonda K taramasi ---
import csv, statistics as st
def load(nu, wr, field='matched_acc'):
    acc = {k: [] for k in KS}
    for s in SEEDS:
        f = f'{CKDIR}/matchedv1{cfgtag(nu,wr)}_{MODE}_s{s}.csv'
        if not os.path.exists(f): continue
        for r in csv.DictReader(open(f)):
            k = int(r['K'])
            if k in acc: acc[k].append(float(r[field]))
    return acc

for field, unit in (('matched_acc','%'), ('matched_logp','nat')):
    print(f'\n--- {field} ({unit}) --- ortalama [min-max] | n={len(SEEDS)} seed x {MP_TRIALS} deneme')
    print(f"{'kol':>22} " + ' '.join(f'K={k:<14}' for k in KS))
    for nu, wr, label in ARMS:
        a = load(nu, wr, field); row = []
        for k in KS:
            row.append(f'{st.mean(a[k]):6.1f} [{min(a[k]):.0f}-{max(a[k]):.0f}]' if a[k] else '     —        ')
        print(f'{label:>22} ' + ' '.join(row))
print('\n(sans %3.3. Bu tablo BETIMLEYICI — hukum yukaridaki birincil metrikte verildi.')
print(' §26b bu sondada seed-basi %8.5-69.5 yayilim olctu; tek basina yorumlanmaz.)')